# CSE 555: Statistical Data Analysis — Final Project

## Communities and Crime Analysis

**Author:** Serkan Eren  
**Dataset:** UCI Communities and Crime  
**Working set:** 43 features selected by label-blind hierarchical clustering on the absolute Pearson correlation distance (see `feature_selection.py`).  
**Target:** `ViolentCrimesPerPop` discretised into 3 classes (Low / Medium / High) by tertiles.

All figures are written to `../figures/`.

In [1]:
"""
CSE 555 Final Project - Communities and Crime Analysis
Author: Serkan Eren
"""
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans, DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import IsolationForest
from collections import Counter
from minisom import MiniSom

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
FIG = os.path.join(ROOT, "figures")
os.makedirs(FIG, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight"})


## 0. Load dataset

In [2]:
attr_pattern = re.compile(r"^@attribute\s+(\S+)\s+\S+", re.IGNORECASE)
names_path = os.path.join(ROOT, "communities.names")
data_path = os.path.join(ROOT, "communities.data")

attr_names = []
with open(names_path) as fh:
    for line in fh:
        m = attr_pattern.match(line.strip())
        if m:
            attr_names.append(m.group(1))

df_full = pd.read_csv(data_path, header=None, names=attr_names, na_values="?")
print(f"Loaded dataset: {df_full.shape}, attributes: {len(attr_names)}")

# Drop non-predictive identifiers
non_predictive = ["state", "county", "community", "communityname", "fold"]
df_full = df_full.drop(columns=non_predictive)

TARGET = "ViolentCrimesPerPop"

# Working set of 43 features obtained by a label-blind, data-driven
# hierarchical-clustering pipeline on the |Pearson r| distance over the
# 100 low-missingness predictors. The authoritative selection (including the
# 12 documented interpretability overrides) is produced by
# notebook/feature_selection.py and written to results_feature_selection.json;
# the list below is exactly that set, merely re-grouped by conceptual block for
# the box-plot panels and the correlation heatmap. A run-time assertion against
# the JSON (see below) guarantees the two stay in sync, so the same
# deterministic selection feeds every subsequent analysis.
feat_blocks = [
    ("Demography",      ["population", "householdsize", "agePct12t29",
                          "agePct65up", "racePctAsian", "racePctHisp"]),
    ("Urbanization",    ["pctUrban", "PopDens", "PctUsePubTrans", "LandArea"]),
    ("Income / wealth", ["medIncome", "blackPerCap", "HispPerCap",
                          "AsianPerCap", "indianPerCap", "OtherPerCap"]),
    ("Housing cost",    ["MedOwnCostPctInc", "MedOwnCostPctIncNoMtg",
                          "MedRentPctHousInc"]),
    ("Education",       ["PctBSorMore"]),
    ("Employment",      ["PctEmploy", "PctEmplProfServ", "PctEmplManu",
                          "pctWFarmSelf"]),
    ("Poverty",         ["PctPopUnderPov"]),
    ("Family",          ["PctKids2Par", "PctIlleg", "MalePctNevMarr",
                          "PctWorkMom"]),
    ("Immigration",     ["NumImmig", "PctForeignBorn", "PctImmigRec8"]),
    ("Housing quality", ["PctHousOwnOcc", "MedNumBR", "PctVacantBoarded",
                          "PctHousOccup", "PctVacMore6Mos", "MedYrHousBuilt",
                          "PctWOFullPlumb", "PctPersDenseHous"]),
    ("Mobility",        ["PctBornSameState", "PctSameHouse85"]),
    ("Policing",        ["LemasPctOfficDrugUn"]),
]
selected = [f for _, feats in feat_blocks for f in feats]
assert len(selected) == 43, f"expected 43 features, got {len(selected)}"

# Consistency guard: the working set MUST equal the authoritative selection
# emitted by feature_selection.py. If the JSON is present (i.e. the selection
# pipeline has been run), assert exact agreement so the report's claim that the
# same deterministic pipeline drives every analysis cannot silently drift.
_fs_json = os.path.join(ROOT, "results_feature_selection.json")
if os.path.exists(_fs_json):
    with open(_fs_json) as _fh:
        _final = set(json.load(_fh)["final_features"])
    assert set(selected) == _final, (
        "feature mismatch between analysis.py and feature_selection.py:\n"
        f"  only in analysis.py: {sorted(set(selected) - _final)}\n"
        f"  only in selection  : {sorted(_final - set(selected))}\n"
        "Re-run feature_selection.py or reconcile the two lists.")
    print("Feature set verified against results_feature_selection.json (43/43).")
else:
    print("results_feature_selection.json not found; run feature_selection.py "
          "to verify the selection. Proceeding with the in-file working set.")

missing = df_full[selected + [TARGET]].isna().sum()
print("Missing values per selected feature:\n", missing)

df = df_full[selected + [TARGET]].copy()
# Median imputation for any remaining missing values
df[selected] = df[selected].fillna(df[selected].median())
df = df.dropna(subset=[TARGET])


Loaded dataset: (1994, 128), attributes: 128
Feature set verified against results_feature_selection.json (43/43).
Missing values per selected feature:
 population               0
householdsize            0
agePct12t29              0
agePct65up               0
racePctAsian             0
racePctHisp              0
pctUrban                 0
PopDens                  0
PctUsePubTrans           0
LandArea                 0
medIncome                0
blackPerCap              0
HispPerCap               0
AsianPerCap              0
indianPerCap             0
OtherPerCap              1
MedOwnCostPctInc         0
MedOwnCostPctIncNoMtg    0
MedRentPctHousInc        0
PctBSorMore              0
PctEmploy                0
PctEmplProfServ          0
PctEmplManu              0
pctWFarmSelf             0
PctPopUnderPov           0
PctKids2Par              0
PctIlleg                 0
MalePctNevMarr           0
PctWorkMom               0
NumImmig                 0
PctForeignBorn           0
PctImmigRec

## Discretize target into 3 classes by tertiles -> Low / Medium / High

In [3]:
labels = ["Low", "Medium", "High"]
df["CrimeClass"] = pd.qcut(df[TARGET], q=3, labels=labels)
class_counts = df["CrimeClass"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)

X = df[selected].values
y = df["CrimeClass"].astype(str).values
classes = labels
class_colors = {"Low": "#2ca02c", "Medium": "#ff7f0e", "High": "#d62728"}
color_arr = np.array([class_colors[c] for c in y])

# Save a small summary used in the report
summary = {
    "n_samples": int(df.shape[0]),
    "n_features": len(selected),
    "feature_names": selected,
    "class_counts": {str(k): int(v) for k, v in class_counts.items()},
}
with open(os.path.join(ROOT, "summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2)



Class distribution:
 CrimeClass
Low       679
Medium    662
High      653
Name: count, dtype: int64


## Quick EDA preview

In [4]:
print('Shape:', df.shape)
print('\nClass counts:')
print(df['CrimeClass'].value_counts().sort_index())
print('\nHead:')
display(df[selected + ['CrimeClass']].head())
print('\nDescribe:')
display(df[selected].describe().round(3))


Shape: (1994, 45)

Class counts:
CrimeClass
Low       679
Medium    662
High      653
Name: count, dtype: int64

Head:


,population,householdsize,agePct12t29,agePct65up,racePctAsian,racePctHisp,pctUrban,PopDens,PctUsePubTrans,LandArea,...,PctVacantBoarded,PctHousOccup,PctVacMore6Mos,MedYrHousBuilt,PctWOFullPlumb,PctPersDenseHous,PctBornSameState,PctSameHouse85,LemasPctOfficDrugUn,CrimeClass
0,0.19,0.33,0.47,0.32,0.12,0.17,1.0,0.26,0.20,0.12,...,0.05,0.71,0.26,0.65,0.06,0.09,0.42,0.50,0.32,Medium
1,0.00,0.16,0.59,0.27,0.45,0.07,1.0,0.12,0.45,0.02,...,0.02,0.79,0.25,0.65,0.00,0.20,0.50,0.34,0.00,High
2,0.00,0.42,0.47,0.32,0.17,0.04,0.0,0.21,0.02,0.01,...,0.29,0.86,0.30,0.52,0.45,0.15,0.49,0.54,0.00,High
3,0.04,0.77,0.50,0.21,0.12,0.10,1.0,0.39,0.28,0.02,...,0.60,0.97,0.47,0.52,0.11,0.12,0.30,0.73,0.00,Medium
4,0.01,0.55,0.38,0.36,0.09,0.05,0.9,0.09,0.02,0.04,...,0.04,0.89,0.55,0.73,0.14,0.02,0.72,0.64,0.00,Low



Describe:


,population,householdsize,agePct12t29,agePct65up,racePctAsian,racePctHisp,pctUrban,PopDens,PctUsePubTrans,LandArea,...,MedNumBR,PctVacantBoarded,PctHousOccup,PctVacMore6Mos,MedYrHousBuilt,PctWOFullPlumb,PctPersDenseHous,PctBornSameState,PctSameHouse85,LemasPctOfficDrugUn
count,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,...,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000,1994.000
mean,0.058,0.463,0.494,0.423,0.154,0.144,0.696,0.233,0.162,0.065,...,0.315,0.205,0.720,0.433,0.494,0.243,0.186,0.609,0.535,0.094
std,0.127,0.164,0.144,0.179,0.209,0.232,0.445,0.203,0.229,0.109,...,0.255,0.218,0.194,0.189,0.232,0.206,0.210,0.204,0.181,0.240
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.010,0.350,0.410,0.300,0.040,0.010,0.000,0.100,0.020,0.020,...,0.000,0.060,0.630,0.290,0.350,0.100,0.060,0.470,0.420,0.000
50%,0.020,0.440,0.480,0.420,0.070,0.040,1.000,0.170,0.070,0.040,...,0.500,0.130,0.770,0.420,0.520,0.190,0.110,0.630,0.540,0.000
75%,0.050,0.540,0.540,0.530,0.170,0.160,1.000,0.280,0.190,0.070,...,0.500,0.270,0.860,0.560,0.670,0.330,0.220,0.778,0.660,0.000
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,...,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


## Task 1 - Boxplots + outlier detection (multivariate Isolation Forest)

In [5]:
print("\n[Task 1] Boxplots and outlier detection ...")

# Split features into two visual panels so 43 boxes remain readable.
# Block boundaries from feat_blocks: panel 1 = first 6 blocks (~22 features),
# panel 2 = remaining blocks (~21 features).
PANEL_SPLIT = 6
panel1_blocks = feat_blocks[:PANEL_SPLIT]
panel2_blocks = feat_blocks[PANEL_SPLIT:]
panel1_feats = [f for _, fs in panel1_blocks for f in fs]
panel2_feats = [f for _, fs in panel2_blocks for f in fs]

def two_panel_boxplot(data, title, fname):
    fig, axes = plt.subplots(2, 1, figsize=(13, 9))
    for ax, panel_feats, panel_blocks in [
        (axes[0], panel1_feats, panel1_blocks),
        (axes[1], panel2_feats, panel2_blocks),
    ]:
        data[panel_feats].boxplot(ax=ax, rot=70, grid=False)
        ax.set_ylabel("Normalized value")
        ax.set_ylim(-0.05, 1.05)
        # block separators
        cum = 0
        for name, fs in panel_blocks[:-1]:
            cum += len(fs)
            ax.axvline(cum + 0.5, color="lightgrey", linewidth=0.7,
                       linestyle="--")
        # block labels
        cum = 0
        for name, fs in panel_blocks:
            mid = cum + len(fs) / 2 + 0.5
            ax.text(mid, 1.07, name, ha="center", va="bottom",
                    fontsize=8, color="#444", transform=ax.get_xaxis_transform())
            cum += len(fs)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname))
    plt.close(fig)

two_panel_boxplot(df, "Feature distributions (before outlier removal)",
                  "boxplot_before.png")



[Task 1] Boxplots and outlier detection ...


## Outlier-method comparison: per-feature IQR (rejected) vs Isolation Forest

In [6]:
# (a) Per-feature IQR with k = 3. Any row that is outside [Q1 - 3*IQR,
#     Q3 + 3*IQR] in any single feature is dropped. Reported only as a
#     baseline to motivate the multivariate alternative.
q1 = df[selected].quantile(0.25)
q3 = df[selected].quantile(0.75)
iqr = q3 - q1
lo = q1 - 3.0 * iqr
hi = q3 + 3.0 * iqr
mask_iqr = ((df[selected] >= lo) & (df[selected] <= hi)).all(axis=1)
n_iqr_removed = int((~mask_iqr).sum())
iqr_class_after = df.loc[mask_iqr, "CrimeClass"].value_counts().sort_index()
print(f"Per-feature IQR (k=3) would remove {n_iqr_removed} rows "
      f"({n_iqr_removed/len(df)*100:.1f}%)")
print("  class distribution under IQR rule:\n",
      iqr_class_after.to_string().replace("\n", "\n  "))

# (b) Multivariate Isolation Forest. Each random tree partitions the feature
#     space; anomalies are isolated with unusually short average paths.
#     contamination=0.05 flags the top 5% globally most anomalous samples.
iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
iso_pred = iso.fit_predict(df[selected].values)  # +1 inlier, -1 outlier
inside = iso_pred == 1
n_removed = int((~inside).sum())
print(f"IsolationForest removed {n_removed} rows "
      f"({n_removed/len(df)*100:.1f}%)")

df_clean = df[inside].reset_index(drop=True)
print(f"Cleaned dataset shape: {df_clean.shape}")
print("Class distribution after Isolation Forest cleaning:\n",
      df_clean["CrimeClass"].value_counts().sort_index())

two_panel_boxplot(df_clean,
                  "Feature distributions after Isolation Forest cleaning (contamination = 0.05)",
                  "boxplot_after.png")

# Use cleaned dataset for the rest of the analysis
X = df_clean[selected].values
y = df_clean["CrimeClass"].astype(str).values
classes = labels
color_arr = np.array([class_colors[c] for c in y])

print("\nDone Task 1.")


Per-feature IQR (k=3) would remove 730 rows (36.6%)
  class distribution under IQR rule:
 CrimeClass
  Low       524
  Medium    445
  High      295


IsolationForest removed 100 rows (5.0%)
Cleaned dataset shape: (1894, 45)
Class distribution after Isolation Forest cleaning:
 CrimeClass
Low       660
Medium    649
High      585
Name: count, dtype: int64



Done Task 1.


## Task 2 - Correlation analysis

In [7]:
print("\n[Task 2] Correlation analysis ...")
# Encode class as ordinal for correlation with class
class_to_int = {"Low": 0, "Medium": 1, "High": 2}
y_int = np.array([class_to_int[c] for c in y])
corr_df = pd.DataFrame(X, columns=selected)
corr_df["CrimeClass"] = y_int
corr_matrix = corr_df.corr(method="pearson")

# With 43 features the matrix is 44x44, so numeric annotations are illegible.
# We drop them and instead reveal block structure by drawing separators at the
# conceptual-block boundaries (the feature order is already the block order).
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, square=True,
            annot=False, cbar_kws={"shrink": 0.6, "label": "Pearson r"},
            xticklabels=True, yticklabels=True, ax=ax)
ax.tick_params(axis="both", labelsize=6)
# block boundaries (cumulative feature counts)
cum = 0
for _, fs in feat_blocks[:-1]:
    cum += len(fs)
    ax.axhline(cum, color="black", linewidth=0.5)
    ax.axvline(cum, color="black", linewidth=0.5)
# separator before the class row/col
ax.axhline(len(selected), color="black", linewidth=1.2)
ax.axvline(len(selected), color="black", linewidth=1.2)
ax.set_title("Pearson correlation matrix (43 features + ordinal class)\n"
             "black lines mark conceptual-block boundaries")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "correlation_heatmap.png"))
plt.close(fig)

class_corr = corr_matrix["CrimeClass"].drop("CrimeClass").sort_values(key=lambda s: s.abs(), ascending=False)
print("Top |correlations| with class (Pearson):\n", class_corr.head(10))

# Spearman is more appropriate for an ordinal class label; report both.
spearman_class = pd.Series(
    {f: stats.spearmanr(X[:, i], y_int).correlation for i, f in enumerate(selected)}
).sort_values(key=lambda s: s.abs(), ascending=False)
print("\nTop |correlations| with class (Spearman):\n", spearman_class.head(10))
# Sanity: rank agreement between Pearson and Spearman feature-class correlations
rank_agree = stats.spearmanr(class_corr.abs(), spearman_class.reindex(class_corr.index).abs()).correlation
print(f"Pearson vs Spearman |corr| rank agreement (Spearman): {rank_agree:.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
class_corr.plot(kind="barh", ax=ax,
                color=["#d62728" if v > 0 else "#1f77b4" for v in class_corr.values])
ax.invert_yaxis()
ax.set_xlabel("Pearson correlation with crime class (ordinal)")
ax.set_title("Per-feature correlation with class label")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "feature_class_correlation.png"))
plt.close(fig)
print("Done Task 2.")



[Task 2] Correlation analysis ...


Top |correlations| with class (Pearson):
 PctKids2Par        -0.685472
PctIlleg            0.622215
PctPopUnderPov      0.516684
PctPersDenseHous    0.466812
PctHousOwnOcc      -0.458732
medIncome          -0.450884
PctVacantBoarded    0.375612
PctHousOccup       -0.354827
MedNumBR           -0.350589
PctBSorMore        -0.348322
Name: CrimeClass, dtype: float64

Top |correlations| with class (Spearman):
 PctKids2Par        -0.700920
PctIlleg            0.684729
PctPersDenseHous    0.606960
PctPopUnderPov      0.572438
PctHousOwnOcc      -0.465576
medIncome          -0.446040
PctHousOccup       -0.414461
PctVacantBoarded    0.384346
MedNumBR           -0.349041
PctEmploy          -0.343874
dtype: float64
Pearson vs Spearman |corr| rank agreement (Spearman): 0.969


Done Task 2.


## Task 3 - Z-score normalization + Fisher Distance per feature

In [8]:
print("\n[Task 3] Z-score + Fisher Distance per feature ...")
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

def fisher_distance_per_feature(Xmat, labels_arr):
    """Multi-class Fisher score: J = trace(S_B) / trace(S_W) per feature."""
    classes_local = np.unique(labels_arr)
    overall_mean = Xmat.mean(axis=0)
    sb = np.zeros(Xmat.shape[1])
    sw = np.zeros(Xmat.shape[1])
    for c in classes_local:
        Xc = Xmat[labels_arr == c]
        nc = Xc.shape[0]
        mu_c = Xc.mean(axis=0)
        sb += nc * (mu_c - overall_mean) ** 2
        sw += ((Xc - mu_c) ** 2).sum(axis=0)
    return sb / np.where(sw == 0, 1e-12, sw)

fisher_raw = fisher_distance_per_feature(Xz, y)
fisher_series = pd.Series(fisher_raw, index=selected).sort_values(ascending=False)
print("Fisher Distance (z-scored features):\n", fisher_series)

fig, ax = plt.subplots(figsize=(8, 9))
fisher_series.plot(kind="barh", ax=ax, color="#3a7bb0")
ax.invert_yaxis()
ax.tick_params(axis="y", labelsize=7)
ax.set_xlabel("Fisher Distance  (trace(S_B)/trace(S_W))")
ax.set_title("Per-feature Fisher Distance (z-scored, 43 features)")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fisher_features.png"))
plt.close(fig)
print("Done Task 3.")



[Task 3] Z-score + Fisher Distance per feature ...
Fisher Distance (z-scored features):
 PctKids2Par              0.896823
PctIlleg                 0.694608
PctPopUnderPov           0.373765
PctPersDenseHous         0.279316
PctHousOwnOcc            0.267834
medIncome                0.255289
PctVacantBoarded         0.182760
PctHousOccup             0.144358
MedNumBR                 0.141277
PctBSorMore              0.139321
PctEmploy                0.127660
racePctHisp              0.122062
MedRentPctHousInc        0.109875
PctWOFullPlumb           0.097383
blackPerCap              0.093504
population               0.092371
HispPerCap               0.089300
LemasPctOfficDrugUn      0.088116
PctSameHouse85           0.070107
PctImmigRec8             0.068631
NumImmig                 0.060650
MalePctNevMarr           0.044442
AsianPerCap              0.036571
PopDens                  0.033066
LandArea                 0.028139
agePct12t29              0.025555
PctForeignBorn           0

Done Task 3.


## Task 4 - PCA + Fisher Distance per PC

In [9]:
print("\n[Task 4] PCA and Fisher Distance per PC ...")
pca = PCA(n_components=Xz.shape[1])
Xpca = pca.fit_transform(Xz)
eigvals = pca.explained_variance_
expl_ratio = pca.explained_variance_ratio_

fisher_pcs = fisher_distance_per_feature(Xpca, y)
pc_names = [f"PC{i+1}" for i in range(Xz.shape[1])]
pc_df = pd.DataFrame({"eigenvalue": eigvals,
                      "explained_var_ratio": expl_ratio,
                      "fisher": fisher_pcs}, index=pc_names)
print(pc_df)

corr_eig_fisher = np.corrcoef(eigvals, fisher_pcs)[0, 1]
print(f"Pearson corr(eigenvalue, Fisher of PC) = {corr_eig_fisher:.3f}")
spearman_eig_fisher = stats.spearmanr(eigvals, fisher_pcs).correlation
print(f"Spearman corr(eigenvalue, Fisher of PC) = {spearman_eig_fisher:.3f}")

# Comparison bar chart. Share the y-axis so the two panels are directly
# comparable: the per-feature signal is spread over several bars, whereas the
# per-PC signal is concentrated almost entirely in PC1.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
fisher_series.plot(kind="bar", ax=axes[0], color="#3a7bb0")
axes[0].set_title("Fisher Distance per original (z-scored) feature")
axes[0].set_ylabel("Fisher Distance")
axes[0].tick_params(axis="x", rotation=90, labelsize=6)
pc_df["fisher"].plot(kind="bar", ax=axes[1], color="#c0504d")
axes[1].set_title("Fisher Distance per principal component")
axes[1].tick_params(axis="x", rotation=90, labelsize=6)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fisher_comparison.png"))
plt.close(fig)

# Eigenvalue vs Fisher scatter
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(eigvals, fisher_pcs, color="#444444")
for i, name in enumerate(pc_names):
    ax.annotate(name, (eigvals[i], fisher_pcs[i]), fontsize=7,
                xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Eigenvalue (variance of PC)")
ax.set_ylabel("Fisher Distance of PC")
ax.set_title(f"Eigenvalue vs Fisher Distance  (Pearson={corr_eig_fisher:.2f}, "
             f"Spearman={spearman_eig_fisher:.2f})")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "eigval_vs_fisher.png"))
plt.close(fig)

# Scree
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(expl_ratio) + 1), expl_ratio, color="#3a7bb0", alpha=0.8)
ax.plot(range(1, len(expl_ratio) + 1), np.cumsum(expl_ratio),
        color="#d62728", marker="o", label="cumulative")
ax.set_xlabel("Principal component index")
ax.set_ylabel("Explained variance ratio")
ax.set_title("PCA scree plot")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG, "pca_scree.png"))
plt.close(fig)
print("Done Task 4.")



[Task 4] PCA and Fisher Distance per PC ...
      eigenvalue  explained_var_ratio    fisher
PC1     8.009659             0.186173  0.560667
PC2     6.147309             0.142885  0.061223
PC3     3.683749             0.085623  0.009583
PC4     2.753335             0.063997  0.016341
PC5     2.381448             0.055353  0.034749
PC6     2.170964             0.050461  0.002918
PC7     2.083353             0.048424  0.036513
PC8     1.481149             0.034427  0.007705
PC9     1.098700             0.025538  0.037206
PC10    1.083406             0.025182  0.002197
PC11    0.938335             0.021810  0.002732
PC12    0.894826             0.020799  0.001856
PC13    0.826212             0.019204  0.002374
PC14    0.764612             0.017772  0.023411
PC15    0.700492             0.016282  0.000865
PC16    0.667908             0.015525  0.001761
PC17    0.603824             0.014035  0.003608
PC18    0.588575             0.013681  0.002820
PC19    0.563487             0.013097  0.00

Done Task 4.


## Task 5 - Scatter plots of top-2 / bottom-2 PCs

In [10]:
print("\n[Task 5] Scatter plots of PCs ...")
def scatter_pcs(idx1, idx2, fname, title):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for c in classes:
        mask = (y == c)
        ax.scatter(Xpca[mask, idx1], Xpca[mask, idx2],
                   s=14, alpha=0.65, label=c, color=class_colors[c],
                   edgecolors="none")
    ax.set_xlabel(f"PC{idx1+1}  (var={expl_ratio[idx1]*100:.1f}%)")
    ax.set_ylabel(f"PC{idx2+1}  (var={expl_ratio[idx2]*100:.1f}%)")
    ax.set_title(title)
    ax.legend(title="Crime class")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname))
    plt.close(fig)

scatter_pcs(0, 1, "pca_top2.png", "Projection onto the two most important PCs")
n = Xz.shape[1]
scatter_pcs(n - 2, n - 1, "pca_bottom2.png", "Projection onto the two least important PCs")
print("Done Task 5.")



[Task 5] Scatter plots of PCs ...


Done Task 5.


## Task 6 - LDA and class-separability metric

In [11]:
print("\n[Task 6] LDA + separability metrics ...")
lda = LinearDiscriminantAnalysis(n_components=2)
Xlda = lda.fit_transform(Xz, y)

fig, ax = plt.subplots(figsize=(7, 5.5))
for c in classes:
    mask = (y == c)
    ax.scatter(Xlda[mask, 0], Xlda[mask, 1], s=14, alpha=0.7,
               label=c, color=class_colors[c], edgecolors="none")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_title("LDA 2D projection")
ax.legend(title="Crime class")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "lda_scatter.png"))
plt.close(fig)

# Quantitative separability: silhouette and KNN cross-val accuracy on the 2D
def class_silhouette(Xmat, labels_arr):
    return silhouette_score(Xmat, labels_arr)

def knn_cv_accuracy(Xmat, labels_arr, k=5, folds=5):
    clf = KNeighborsClassifier(n_neighbors=k)
    return cross_val_score(clf, Xmat, labels_arr, cv=folds, scoring="accuracy").mean()

sep = {
    "PCA-2D silhouette": class_silhouette(Xpca[:, :2], y),
    "LDA-2D silhouette": class_silhouette(Xlda, y),
    "PCA-2D 5NN accuracy (5-fold)": knn_cv_accuracy(Xpca[:, :2], y),
    "LDA-2D 5NN accuracy (5-fold)": knn_cv_accuracy(Xlda, y),
}
print("Separability metrics:")
for k, v in sep.items():
    print(f"  {k}: {v:.3f}")
print("Done Task 6.")



[Task 6] LDA + separability metrics ...


Separability metrics:
  PCA-2D silhouette: 0.049
  LDA-2D silhouette: 0.127
  PCA-2D 5NN accuracy (5-fold): 0.569
  LDA-2D 5NN accuracy (5-fold): 0.659
Done Task 6.


## Task 7 - Clustering: K-Means, DBSCAN on PCA(2); t-SNE and SOM on original

In [12]:
print("\n[Task 7] Clustering ...")
X2 = Xpca[:, :2]

km = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km.fit_predict(X2)

db = DBSCAN(eps=0.5, min_samples=10)
db_labels = db.fit_predict(X2)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_db_noise = int((db_labels == -1).sum())
print(f"DBSCAN: {n_db_clusters} clusters, {n_db_noise} noise points")

tsne = TSNE(n_components=2, perplexity=30, init="pca",
            learning_rate="auto", random_state=42)
Xtsne = tsne.fit_transform(Xz)

# SOM on original (z-scored) data
som_size = 12
som = MiniSom(som_size, som_size, Xz.shape[1], sigma=1.5,
              learning_rate=0.5, random_seed=42)
som.random_weights_init(Xz)
som.train_random(Xz, 5000, verbose=False)
som_coords = np.array([som.winner(x) for x in Xz])  # (N,2)
# Add jitter so overlapping points are visible (for the scatter view)
jitter = np.random.normal(0, 0.25, size=som_coords.shape)
som_xy = som_coords + jitter

# Per-cell dominant class and U-matrix-style summaries for the SOM hit-map view
som_winner_class = {}
som_hit_count = np.zeros((som_size, som_size), dtype=int)
for (i, j), cls in zip(map(tuple, som_coords), y):
    som_winner_class.setdefault((i, j), []).append(cls)
    som_hit_count[i, j] += 1

dominant = np.full((som_size, som_size), -1)
purity = np.zeros((som_size, som_size))
for (i, j), cls_list in som_winner_class.items():
    c = Counter(cls_list)
    top_cls, top_n = c.most_common(1)[0]
    dominant[i, j] = labels.index(top_cls)
    purity[i, j] = top_n / len(cls_list)

def plot_clusters(Xmat, cluster_labels, true_labels, fname, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for c in classes:
        mask = (true_labels == c)
        axes[0].scatter(Xmat[mask, 0], Xmat[mask, 1], s=14, alpha=0.7,
                        label=c, color=class_colors[c], edgecolors="none")
    axes[0].legend(title="True class")
    axes[0].set_title(title + " - true classes")
    cl_unique = np.unique(cluster_labels)
    palette = sns.color_palette("tab10", n_colors=max(3, len(cl_unique)))
    for i, cl in enumerate(cl_unique):
        mask = (cluster_labels == cl)
        lbl = "noise" if cl == -1 else f"cluster {cl}"
        col = "#999999" if cl == -1 else palette[i % len(palette)]
        axes[1].scatter(Xmat[mask, 0], Xmat[mask, 1], s=14, alpha=0.7,
                        label=lbl, color=col, edgecolors="none")
    axes[1].legend(title="Cluster", fontsize=8)
    axes[1].set_title(title + " - clusters")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname))
    plt.close(fig)

plot_clusters(X2, km_labels, y, "cluster_kmeans.png", "K-Means on PCA(2)")
plot_clusters(X2, db_labels, y, "cluster_dbscan.png", "DBSCAN on PCA(2)")
# For t-SNE / SOM, plot true classes only (no clustering algorithm applied
# beyond the embedding itself, per the assignment wording)
fig, ax = plt.subplots(figsize=(7, 5.5))
for c in classes:
    mask = (y == c)
    ax.scatter(Xtsne[mask, 0], Xtsne[mask, 1], s=14, alpha=0.7,
               label=c, color=class_colors[c], edgecolors="none")
ax.set_title("t-SNE embedding of original data (true classes)")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.legend(title="Crime class")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "cluster_tsne.png"))
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Panel 1: scatter of winners coloured by true class (jittered for visibility)
for c in classes:
    mask = (y == c)
    axes[0].scatter(som_xy[mask, 0], som_xy[mask, 1], s=14, alpha=0.7,
                    label=c, color=class_colors[c], edgecolors="none")
axes[0].set_title("SOM winners (true class)")
axes[0].set_xlabel("SOM x"); axes[0].set_ylabel("SOM y")
axes[0].legend(title="Crime class", fontsize=8)

# Panel 2: per-cell hit count (how many samples landed on each neuron)
im2 = axes[1].imshow(som_hit_count.T, origin="lower", cmap="viridis", aspect="equal")
axes[1].set_title("SOM hit count per neuron")
axes[1].set_xlabel("SOM x"); axes[1].set_ylabel("SOM y")
fig.colorbar(im2, ax=axes[1], fraction=0.046)

# Panel 3: dominant class per cell, with purity as alpha
from matplotlib.colors import ListedColormap
cmap_dom = ListedColormap(["#f0f0f0"] + [class_colors[c] for c in labels])  # -1 -> light grey
display_grid = dominant + 1  # shift -1 (empty) to 0
axes[2].imshow(display_grid.T, origin="lower", cmap=cmap_dom,
               vmin=0, vmax=len(labels), aspect="equal", alpha=1.0)
# overlay purity as text in non-empty cells
for i in range(som_size):
    for j in range(som_size):
        if dominant[i, j] >= 0:
            axes[2].text(i, j, f"{int(purity[i,j]*100)}", ha="center",
                         va="center", fontsize=6, color="white")
axes[2].set_title("Dominant class per neuron (% purity)")
axes[2].set_xlabel("SOM x"); axes[2].set_ylabel("SOM y")
# legend patches
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=class_colors[c], label=c) for c in labels]
handles.append(mpatches.Patch(color="#f0f0f0", label="empty"))
axes[2].legend(handles=handles, fontsize=7, loc="lower right")

fig.tight_layout()
fig.savefig(os.path.join(FIG, "cluster_som.png"))
plt.close(fig)

# Quantify how class-organised the SOM is
nonempty = dominant.flatten() >= 0
mean_purity = float(purity.flatten()[nonempty].mean())
print(f"SOM mean cell purity (non-empty cells): {mean_purity:.3f}")

# Cluster vs class agreement
cluster_metrics = {
    "KMeans ARI": adjusted_rand_score(y_int, km_labels),
    "KMeans NMI": normalized_mutual_info_score(y_int, km_labels),
    "DBSCAN ARI": adjusted_rand_score(y_int, db_labels),
    "DBSCAN NMI": normalized_mutual_info_score(y_int, db_labels),
}
print("Cluster agreement:")
for k, v in cluster_metrics.items():
    print(f"  {k}: {v:.3f}")
print("Done Task 7.")



[Task 7] Clustering ...


DBSCAN: 6 clusters, 233 noise points


SOM mean cell purity (non-empty cells): 0.675
Cluster agreement:
  KMeans ARI: 0.142
  KMeans NMI: 0.147
  DBSCAN ARI: 0.018
  DBSCAN NMI: 0.039
Done Task 7.


## Task 8 - Hypothesis test on PctKids2Par mean (Low vs High classes)

In [13]:
print("\n[Task 8] Hypothesis test ...")
feature_for_test = "PctKids2Par"
g_low = df_clean.loc[df_clean["CrimeClass"] == "Low", feature_for_test].values
g_high = df_clean.loc[df_clean["CrimeClass"] == "High", feature_for_test].values

ttest_results = {}
for n in (36, 64):
    rng = np.random.default_rng(42)
    s_low = rng.choice(g_low, size=n, replace=False)
    s_high = rng.choice(g_high, size=n, replace=False)
    t, p = stats.ttest_ind(s_low, s_high, equal_var=False)
    ttest_results[n] = {"t": float(t), "p": float(p),
                        "mean_low": float(s_low.mean()),
                        "mean_high": float(s_high.mean()),
                        "std_low": float(s_low.std(ddof=1)),
                        "std_high": float(s_high.std(ddof=1))}
    print(f"  n={n}: t={t:.3f}, p={p:.3e}, mean_low={s_low.mean():.3f}, mean_high={s_high.mean():.3f}")

# Population (full-class) reference
t_full, p_full = stats.ttest_ind(g_low, g_high, equal_var=False)
print(f"  population: t={t_full:.3f}, p={p_full:.3e}")
print("Done Task 8.")



[Task 8] Hypothesis test ...
  n=36: t=9.011, p=2.849e-13, mean_low=0.759, mean_high=0.463
  n=64: t=12.731, p=7.349e-23, mean_low=0.792, mean_high=0.444


  population: t=39.283, p=2.627e-207
Done Task 8.


## Save metrics for the report

In [14]:
results = {
    "summary": summary,
    "outlier_removed": int(n_removed),
    "class_counts_clean": {k: int(v) for k, v in df_clean["CrimeClass"].value_counts().sort_index().items()},
    "fisher_features": fisher_series.to_dict(),
    "fisher_pcs": pc_df.to_dict(orient="index"),
    "eigval_fisher_pearson": float(corr_eig_fisher),
    "eigval_fisher_spearman": float(spearman_eig_fisher),
    "separability": {k: float(v) for k, v in sep.items()},
    "clustering": {k: float(v) for k, v in cluster_metrics.items()},
    "dbscan_clusters": int(n_db_clusters),
    "dbscan_noise": int(n_db_noise),
    "ttest": ttest_results,
    "ttest_population": {"t": float(t_full), "p": float(p_full)},
    "feature_for_test": feature_for_test,
    "class_corr_top_pearson": class_corr.head(10).to_dict(),
    "class_corr_top_spearman": spearman_class.head(10).to_dict(),
    "pearson_spearman_rank_agreement": float(rank_agree),
    "som_mean_cell_purity": mean_purity,
}
with open(os.path.join(ROOT, "results.json"), "w") as fh:
    json.dump(results, fh, indent=2)
print("\nAll tasks complete. Figures in:", FIG)



All tasks complete. Figures in: /home/se/Desktop/Crime/figures
